In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
PROJECT = '/content/drive/MyDrive/KaburAjaDulu'
PROCESSED = f'{PROJECT}/data'
OUTPUT    = f'{PROJECT}/data'

In [3]:
print(os.path.exists(f'{PROCESSED}/review_low_confidence_labeled.csv'),
      '← review_low_confidence_labeled.csv')
print(os.path.exists(f'{PROCESSED}/high_confidence_audit_labeled.csv'),
      '← high_confidence_audit_labeled.csv')
print(os.path.exists(f'{OUTPUT}/sentiment_results.csv'),
      '← sentiment_results.csv')


True ← review_low_confidence_labeled.csv
True ← high_confidence_audit_labeled.csv
True ← sentiment_results.csv


In [4]:
!pip install scikit-learn -q
import pandas as pd

In [5]:
df_low = pd.read_csv(f'{PROCESSED}/review_low_confidence_labeled.csv', sep=';')
df_low = df_low[['clean_text', 'label_final']].rename(columns={'label_final': 'label'})
df_low['sumber'] = 'manual_low_conf'
print(f"Low confidence (manual): {len(df_low)}")

Low confidence (manual): 2511


In [6]:
df_audit = pd.read_csv(f'{PROCESSED}/high_confidence_audit_labeled.csv', sep=';')
df_audit = df_audit[['clean_text', 'label_final']].rename(columns={'label_final': 'label'})
df_audit['sumber'] = 'manual_audit'
print(f"High confidence audit (manual): {len(df_audit)}")

High confidence audit (manual): 400


In [7]:
df_all_raw = pd.read_csv(f'{OUTPUT}/sentiment_results.csv')
df_high_all = df_all_raw[df_all_raw['confidence'] >= 70].copy()

In [8]:
audit_texts = set(df_audit['clean_text'].tolist())
df_silver = df_high_all[~df_high_all['clean_text'].isin(audit_texts)].copy()
df_silver = df_silver[['clean_text', 'sentiment']].rename(columns={'sentiment': 'label'})
df_silver['sumber'] = 'silver_high_conf'
print(f"Silver label (high conf): {len(df_silver)}")

Silver label (high conf): 15903


In [9]:
from sklearn.model_selection import train_test_split

In [10]:
df_combined = pd.concat([df_low, df_silver, df_audit], ignore_index=True)
df_combined = df_combined[df_combined['clean_text'].notna()]
df_combined = df_combined[df_combined['clean_text'].str.strip() != '']
df_combined = df_combined[df_combined['label'].isin(['negatif', 'netral', 'positif'])]
df_combined = df_combined.drop_duplicates(subset='clean_text')

In [11]:
print(f"\nTotal: {len(df_combined)}")
print("\nDistribusi label:")
print(df_combined['label'].value_counts())
print("\nDistribusi sumber:")
print(df_combined['sumber'].value_counts())


Total: 18814

Distribusi label:
label
negatif    11417
netral      3738
positif     3659
Name: count, dtype: int64

Distribusi sumber:
sumber
silver_high_conf    15903
manual_low_conf      2511
manual_audit          400
Name: count, dtype: int64


In [12]:
label_map = {'negatif': 0, 'netral': 1, 'positif': 2}
df_combined['label_num'] = df_combined['label'].map(label_map)

In [13]:
df_test      = df_combined[df_combined['sumber'] == 'manual_audit'].copy()
df_train_pool = df_combined[df_combined['sumber'] != 'manual_audit'].copy()

df_train, df_val = train_test_split(
    df_train_pool,
    test_size=0.1,
    random_state=42,
    stratify=df_train_pool['label_num']
)

print(f"Training set  : {len(df_train)}")
print(f"Validation set: {len(df_val)}")
print(f"Test set      : {len(df_test)}")


Training set  : 16572
Validation set: 1842
Test set      : 400


In [14]:
os.makedirs(f'{PROCESSED}/fine_tuning', exist_ok=True)

df_train[['clean_text', 'label', 'label_num']].to_csv(
    f'{PROCESSED}/fine_tuning/train.csv', index=False
)
df_val[['clean_text', 'label', 'label_num']].to_csv(
    f'{PROCESSED}/fine_tuning/val.csv', index=False
)
df_test[['clean_text', 'label', 'label_num']].to_csv(
    f'{PROCESSED}/fine_tuning/test.csv', index=False
)

print(f"   {PROCESSED}/fine_tuning/")
print("   - train.csv")
print("   - val.csv")
print("   - test.csv")

   /content/drive/MyDrive/KaburAjaDulu/data/fine_tuning/
   - train.csv
   - val.csv
   - test.csv
